# Web Search with Ollama and Llama 3

In [89]:
import json
from ddgs import DDGS

def web_search(query: str):
    try:
        results = DDGS().text(query, max_results=2)
        
        # print(results)  # Debugging: Print the raw results from DuckDuckGo
        return [
            {
                "title": result.get("title"),
                "link": result.get("href"),
                "snippet": result.get("body")
            }
            for result in results    
        ]
    except Exception as e:
        print(f"Error performing web search: {e}")
        return []
    
TOOLS_MAP = {
    "web_search": web_search
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Perform a web search using DuckDuckGo for current information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to perform"
                    }
                }
            },
            "required": ["query"]
        }    
    }
]    

In [ ]:
import ollama


def llm_search(query):
    
    
    system_prompt = """
        You are a helpful assistant tools:

        Use a tool only for current, real-time, or uncertain facts.
        Respond directly for greetings, math, and general knowledge.
    """

    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    while True:

        response = ollama.chat(
            model='llama3.1', 
            messages=messages, 
            tools=tools, 
            stream=False, 
            options={
                "temperature": 0,
            
            }
        )

        message = response['message']

        if not message.get('tool_calls'):
            return message['content']

        for tool_call in message['tool_calls']:
            name = tool_call['function']['name']
            arguments = tool_call['function']['arguments']
            
            fn = TOOLS_MAP.get(name)
            
            result = fn(**arguments) 
            
            messages.append({"role": "tool", "content": json.dumps(result)})

In [91]:
llm_search("Current top news about AI?")

'The current top news about AI includes:\n\n* Google News has a dedicated section for artificial intelligence, featuring the latest articles and videos.\n* The Wall Street Journal (WSJ) also has a section dedicated to AI news, covering the latest developments in the field.\n\nYou can visit these sources for more information on the latest AI news.'